🧱 **Célula 1 — Cabeçalho e Introdução**

"""

PROJETO: Análise e Automação de Vendas de um E-commerce Brasileiro

AUTOR: Gustavo de Paula Silva (@gustavogit4)

OBJETIVO:
    Este notebook gera um banco de dados relacional realista (clientes, produtos e vendas)
    para demonstrar habilidades em Python, SQL e Power BI, abordando conceitos de:
    
    - ETL e modelagem de dados;
    
    - simulação estatística e controle de reprodutibilidade;
    
    - geração de dados sintéticos representativos para análise exploratória e BI.

"""

⚙️ **Célula 2 — Importação de bibliotecas**

In [44]:
# ============================
# 2️⃣ IMPORTAÇÃO DE BIBLIOTECAS
# ============================

# Instalação do pacote Faker (caso não esteja disponível)
%pip install faker --quiet

# Importações principais
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import date
import sqlite3

**🧩 Célula 3 — Configurações do simulador**

In [45]:
# ============================
# 3️⃣ CONFIGURAÇÕES DO SIMULADOR
# ============================

"""
Define os parâmetros principais da simulação e garante reprodutibilidade.
Todas as variáveis-chave do projeto estão centralizadas aqui para fácil ajuste.
"""

# Parâmetros globais
N_CLIENTES = 1000
N_PRODUTOS = 50
N_VENDAS = 15000
SEED = 42

# Configuração de reprodutibilidade
np.random.seed(SEED)
random.seed(SEED)

# Inicialização do gerador Faker
faker = Faker('pt_BR')

print("Configurações definidas com sucesso ✅")

Configurações definidas com sucesso ✅


**👥 Célula 4 — Geração da Tabela de Clientes**

In [39]:
# ============================
# 4️⃣ GERAÇÃO DA TABELA DE CLIENTES
# ============================

"""
Cria a tabela de clientes com informações demográficas e geográficas simuladas.
Os dados são baseados em distribuições realistas de idade, gênero e cidade,
utilizando o pacote Faker para geração de nomes e datas.
"""

# Distribuição de idades: média 35 anos, desvio padrão 10
idades = np.random.normal(35, 10, N_CLIENTES).astype(int)
idades = np.clip(idades, 18, 70)  # limite entre 18 e 70 anos

# Geração dos dados dos clientes
clientes = pd.DataFrame({
    'id_cliente': range(1, N_CLIENTES + 1),
    'nome': [faker.name() for _ in range(N_CLIENTES)],
    'idade': idades,
    'genero': np.random.choice(['Masculino', 'Feminino'], size=N_CLIENTES),
    'cidade': [faker.city() for _ in range(N_CLIENTES)],
    'data_cadastro': [faker.date_between(start_date='-3y', end_date='today') for _ in range(N_CLIENTES)]
})

# Visualiza as 5 primeiras linhas
clientes.head()

,id_cliente,nome,idade,genero,cidade,data_cadastro
0,1,Ana Júlia Monteiro,39,Masculino,Camargo Paulista,2024-04-29
1,2,Augusto Caldeira,33,Feminino,Monteiro de Minas,2025-04-22
2,3,Noah Marques,41,Masculino,Porto das Pedras,2025-05-14
3,4,Kamilly Pinto,50,Feminino,Costela da Mata,2024-10-04
4,5,Clara Teixeira,32,Masculino,Rodrigues,2025-03-10


**🛍️ Célula 5 — Geração da Tabela de Produtos**

In [42]:
# ============================
# 5️⃣ GERAÇÃO DA TABELA DE PRODUTOS
# ============================

"""
Cria a tabela de produtos com categorias distintas e faixas de preço realistas.
Cada categoria recebe 10 produtos gerados aleatoriamente, com valores compatíveis
com o mercado. Essa estrutura permitirá análises de ticket médio e lucratividade.
"""

# Dicionário com categorias e faixas de preço (mínimo, máximo)
categorias = {
    'Alimentos': (5, 50),
    'Bebidas': (3, 80),
    'Limpeza': (5, 40),
    'Higiene': (8, 60),
    'Eletronicos': (100, 3000)
}

produtos_lista = []
id_produto = 1

# Gera 10 produtos por categoria com preços dentro da faixa especificada
for categoria, (min_preco, max_preco) in categorias.items():
    for _ in range(10):
        produtos_lista.append({
            'id_produto': id_produto,
            'nome_produto': faker.word().capitalize(),
            'categoria': categoria,
            'preco_unitario': round(np.random.uniform(min_preco, max_preco), 2)
        })
        id_produto += 1

# Converte em DataFrame
produtos = pd.DataFrame(produtos_lista)

# Visualiza as 5 primeiras linhas
produtos.head()

,id_produto,nome_produto,categoria,preco_unitario
0,1,Aliquam,Alimentos,32.72
1,2,Cupiditate,Alimentos,13.46
2,3,Ipsa,Alimentos,20.99
3,4,Incidunt,Alimentos,40.27
4,5,Natus,Alimentos,29.94


**🧠 Célula 6 — Funções Auxiliares**

In [43]:
# ============================
# 6️⃣ FUNÇÕES AUXILIARES
# ============================

"""
Esta seção define as funções utilizadas na geração das vendas:
1. gerar_data_venda(): cria uma data de venda entre 2022 e 2024 com sazonalidade.
2. canal_por_idade(): determina o canal de compra com base na faixa etaria do cliente.
"""

from datetime import date

def gerar_data_venda():
    """
    Gera uma data de venda entre 2022 e 2024, reforçando sazonalidade
    em meses de alta demanda (maio, novembro e dezembro).

    Retorna:
        datetime.date: data da venda.
    """
    data_inicio = date(2022, 1, 1)
    data_fim = date(2024, 12, 31)

    data = faker.date_between(start_date=data_inicio, end_date=data_fim)
    mes = data.month

    # Meses de pico: maio (Dia das Maes), novembro (Black Friday), dezembro (Natal)
    if mes in [5, 11, 12] and random.random() < 0.6:
        ano = random.choice([2022, 2023, 2024])
        dia = random.randint(1, 28)
        return date(ano, mes, dia)
    return data


def canal_por_idade(idade):
    """
    Define o canal de venda com base na idade do cliente.

    - Jovens (<30): preferem e-commerce.
    - Adultos (30–49): equilibram loja fisica e online.
    - Idosos (50+): preferem WhatsApp e loja fisica.

    Retorna:
        str: canal de venda.
    """
    if idade < 30:
        return np.random.choice(['E-commerce', 'Loja Fisica', 'WhatsApp'], p=[0.6, 0.3, 0.1])
    elif idade < 50:
        return np.random.choice(['E-commerce', 'Loja Fisica', 'WhatsApp'], p=[0.4, 0.5, 0.1])
    else:
        return np.random.choice(['E-commerce', 'Loja Fisica', 'WhatsApp'], p=[0.2, 0.4, 0.4])

**💸 Célula 7 — Geração da Tabela de Vendas**

In [46]:
# ============================
# 7️⃣ GERAÇÃO DA TABELA DE VENDAS
# ============================

"""
Cria a tabela de vendas integrando as tabelas de clientes e produtos.
Aplica comportamento de compra realista:
- Distribuição desigual de clientes (alguns compram mais que outros)
- Quantidade e valor ajustados conforme categoria
- Sazonalidade e variação por canal de venda
"""

# Probabilidades desiguais de clientes realizarem mais compras
pesos = np.random.exponential(scale=0.8, size=N_CLIENTES)
pesos = pesos / pesos.sum()

clientes_vendas = np.random.choice(
    clientes['id_cliente'],
    size=N_VENDAS,
    p=pesos
)

# Pré-processa dicionários para acesso rápido no loop
idades_clientes = clientes.set_index('id_cliente')['idade'].to_dict()
produtos_dict = produtos.set_index('id_produto')[['categoria', 'preco_unitario']].to_dict('index')

# Cria as vendas
vendas = []

for i in range(N_VENDAS):
    id_cliente = int(clientes_vendas[i])
    idade_cliente = idades_clientes[id_cliente]
    id_produto = random.choice(list(produtos_dict.keys()))
    categoria = produtos_dict[id_produto]['categoria']
    preco = produtos_dict[id_produto]['preco_unitario']

    # Quantidade média depende da categoria do produto
    if categoria == 'Eletronicos':
        quantidade = max(1, int(np.random.normal(1.2, 0.6)))
    elif categoria in ['Alimentos', 'Bebidas']:
        quantidade = max(1, int(np.random.normal(3, 1.2)))
    else:
        quantidade = max(1, int(np.random.normal(2, 0.8)))

    data_venda = gerar_data_venda()
    canal_venda = canal_por_idade(idade_cliente)

    vendas.append({
        'id_venda': i + 1,
        'id_cliente': id_cliente,
        'id_produto': id_produto,
        'quantidade': quantidade,
        'data_venda': data_venda,
        'canal_venda': canal_venda,
        'valor_total': round(quantidade * preco, 2),
        'ano_venda': data_venda.year,
        'mes_venda': data_venda.month
    })

vendas = pd.DataFrame(vendas)

# Inserindo 1% de valores nulos para testes de limpeza
for col in ['canal_venda', 'quantidade']:
    nulos = int(len(vendas) * 0.01)
    vendas.loc[vendas.sample(n=nulos).index, col] = np.nan

# Visualiza as 5 primeiras linhas
vendas.head()

,id_venda,id_cliente,id_produto,quantidade,data_venda,canal_venda,valor_total,ano_venda,mes_venda
0,1,194,41,1.0,2022-08-08,WhatsApp,882.48,2022,8
1,2,519,8,2.0,2022-06-17,E-commerce,13.18,2022,6
2,3,849,2,2.0,2023-06-11,E-commerce,26.92,2023,6
3,4,717,48,1.0,2022-03-30,E-commerce,2152.27,2022,3
4,5,790,18,2.0,2022-05-24,E-commerce,62.72,2022,5


**🔍 Célula 8 — Validação dos Dados**

In [47]:
# ============================
# 8️⃣ VALIDAÇÃO DOS DADOS
# ============================

"""
Verifica a integridade, consistência e estrutura dos dados antes de salvar o banco.
Essa etapa garante que o dataset está limpo, coerente e pronto para uso em análises.
"""

print("🔎 Verificando integridade dos dados...\n")

# 1. Contagem de registros
print(f"Clientes: {len(clientes)} registros")
print(f"Produtos: {len(produtos)} registros")
print(f"Vendas: {len(vendas)} registros\n")

# 2. Checagem de valores nulos
print("Valores nulos por coluna (tabela vendas):")
print(vendas.isnull().sum())
print()

# 3. Tipos de dados e amostra das tabelas
print("Estrutura da tabela de vendas:")
print(vendas.dtypes)
print("\nExemplo de registros:")
display(vendas.head())

# 4. Verificação básica de coerência
vendas_invalidas = vendas[vendas['quantidade'] <= 0]
if len(vendas_invalidas) > 0:
    print(f"Atenção ⚠️: {len(vendas_invalidas)} registros com quantidade inválida detectados.")
else:
    print("✅ Nenhuma quantidade inválida encontrada.\n")

# 5. Análise resumida dos valores monetários
print("Resumo estatístico de valores totais das vendas:")
display(vendas['valor_total'].describe())

🔎 Verificando integridade dos dados...

Clientes: 1000 registros
Produtos: 50 registros
Vendas: 15000 registros

Valores nulos por coluna (tabela vendas):
id_venda         0
id_cliente       0
id_produto       0
quantidade     150
data_venda       0
canal_venda    150
valor_total      0
ano_venda        0
mes_venda        0
dtype: int64

Estrutura da tabela de vendas:
id_venda         int64
id_cliente       int64
id_produto       int64
quantidade     float64
data_venda      object
canal_venda     object
valor_total    float64
ano_venda        int64
mes_venda        int64
dtype: object

Exemplo de registros:


,id_venda,id_cliente,id_produto,quantidade,data_venda,canal_venda,valor_total,ano_venda,mes_venda
0,1,194,41,1.0,2022-08-08,WhatsApp,882.48,2022,8
1,2,519,8,2.0,2022-06-17,E-commerce,13.18,2022,6
2,3,849,2,2.0,2023-06-11,E-commerce,26.92,2023,6
3,4,717,48,1.0,2022-03-30,E-commerce,2152.27,2022,3
4,5,790,18,2.0,2022-05-24,E-commerce,62.72,2022,5


✅ Nenhuma quantidade inválida encontrada.

Resumo estatístico de valores totais das vendas:


,valor_total
count,15000.000000
mean,391.427875
std,786.187475
min,5.110000
25%,36.020000
50%,72.040000
75%,161.080000
max,6456.810000


**💾 Célula 9 — Criação e Salvamento do Banco SQLite**

In [48]:
# ============================
# 9️⃣ CRIAÇÃO E SALVAMENTO DO BANCO SQLITE
# ============================

"""
Cria o banco de dados 'ecommerce_realista.db' e armazena as tabelas
clientes, produtos e vendas. O SQLite é leve, relacional e ideal
para integração com Python, SQL e Power BI.
"""

# Caminho e criação do banco SQLite
conn = sqlite3.connect('ecommerce_realista.db')

# Exporta as tabelas para o banco
clientes.to_sql('clientes', conn, if_exists='replace', index=False)
produtos.to_sql('produtos', conn, if_exists='replace', index=False)
vendas.to_sql('vendas', conn, if_exists='replace', index=False)

# Finaliza e fecha a conexão
conn.commit()
conn.close()

print("✅ Banco de dados 'ecommerce_realista.db' criado e populado com sucesso!")

# Verificação final
import os
if os.path.exists('ecommerce_realista.db'):
    tamanho_mb = os.path.getsize('ecommerce_realista.db') / (1024 * 1024)
    print(f"Tamanho aproximado do arquivo: {tamanho_mb:.2f} MB")
else:
    print("⚠️ Erro: banco de dados não encontrado.")

✅ Banco de dados 'ecommerce_realista.db' criado e populado com sucesso!
Tamanho aproximado do arquivo: 0.82 MB
